# mPES - Colab Launcher desde GitHub

Este notebook clona una copia ligera del repositorio en el almacenamiento local
de Colab y ejecuta la optimizacion desde `h1/`. Los resultados se conservan
en Google Drive.

Configura en la primera celda el repositorio y la rama. El clonado usa
`--depth 1 --single-branch` para reducir tiempo, espacio y trafico de red.
Para `ens_sprb` o `ens_accq`, los modelos pre-entrenados viajan dentro del
propio clon (`h1/ml/pes_{dqn,rdqn,trf}/inputs/*_model.keras`); si en Drive hay
copias mas recientes (`MyDrive/mPES/<pes_dqn|pes_rdqn|pes_trf>/`, donde las
guarda `retrain_gpu.ipynb`), esas tienen prioridad y se copian al clon. Los
modelos nunca se reentrenan desde este notebook.

Ejecuta todas las celdas en orden. Para ejecuciones largas, activa Background
execution en la sesion de Colab.

In [3]:
"""Colab launcher for mPES Bayesian optimisation."""
# Mount Google Drive before cloning the repository.
# ==========================================================================
# MOUNT GOOGLE DRIVE
# ==========================================================================
# pyright: reportMissingImports=false
# pylint: disable=import-error,no-name-in-module
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive', force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os

REPOSITORY   = 'https://github.com/Maximiliano0/mPES_2026.git'
BRANCH       = 'new_uq'
WORKSPACE    = '/content/mPES'
H_DIR        = os.path.join(WORKSPACE, 'h1')
UTILS_DIR    = os.path.join(WORKSPACE, 'utils')
OUTPUT_ROOT  = '/content/drive/MyDrive/mPES/runs'
PKG          = 'ens_sprb'  # ql | dql | dqn | rdqn | ac | tr | ens_sprb | ens_accq
N_TRIALS     = 50
RESUME_DATE  = ''  # YYYY-MM-DD to resume, or '' for a new run
USE_GPU      = 0

valid_packages = ('ql', 'dql', 'dqn', 'rdqn', 'ac', 'tr', 'ens_sprb', 'ens_accq')
if PKG not in valid_packages:
    raise ValueError(f'Unsupported PKG: {PKG!r}')

os.environ.update({
    'DRIVE_DIR': OUTPUT_ROOT,
    'H_DIR': H_DIR,
    'REPO_DIR': WORKSPACE,
    'WORKSPACE_DIR': WORKSPACE,
    'PKG': PKG,
    'N_TRIALS': str(N_TRIALS),
    'RESUME_DATE': RESUME_DATE,
    'MPES_USE_GPU': str(USE_GPU),
    'MPES_MODEL_ROOT': '',
})
print(f'[INFO] [Celda 1] Configuración guardada: PKG={PKG!r}, N_TRIALS={N_TRIALS}, BRANCH={BRANCH!r}')

[INFO] [Celda 1] Configuración guardada: PKG='ens_sprb', N_TRIALS=50, BRANCH='new_uq'


In [10]:
# Shallow clone: only the selected branch and its current snapshot.
import os
import subprocess

if os.path.isdir(WORKSPACE):
    print(f"[INFO] [Celda 2] Borrando workspace anterior en {WORKSPACE}...")
    subprocess.run(['rm', '-rf', WORKSPACE], check=True)

print(f"[INFO] [Celda 2] Clonando el repositorio rama '{BRANCH}'...")
subprocess.run([
    'git', 'clone', '--depth', '1', '--single-branch', '--branch', BRANCH,
    REPOSITORY, WORKSPACE,
], check=True)

print("[INFO] [Celda 2] Ejecutando setup_colab.sh (Instalación de dependencias)...")
setup = subprocess.run(
    ['bash', os.path.join(H_DIR, 'general', 'colab', 'setup_colab.sh')],
    check=False,
    env=os.environ.copy(),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(setup.stdout)
if setup.returncode != 0:
    raise RuntimeError(f'setup_colab.sh exited with code {setup.returncode}')
print(f'[INFO] [Celda 2] Shallow clone y setup completados: {REPOSITORY}@{BRANCH}')

[INFO] [Celda 2] Borrando workspace anterior en /content/mPES...
[INFO] [Celda 2] Clonando el repositorio rama 'new_uq'...
[INFO] [Celda 2] Ejecutando setup_colab.sh (Instalación de dependencias)...

  mPES  Colab Pro+ bootstrap

 Drive workspace: /content/drive/MyDrive/mPES/runs

  Installing Python dependencies

  Checking pinned optimisation and training dependencies
  Required runtime packages already match project versions.

  Exporting mPES environment variables

 Env vars sourced from /content/mpes_env.sh

  Bootstrap complete. Next: run utils/colab/run_colab.sh <PKG> <TRIALS>


[INFO] [Celda 2] Shallow clone y setup completados: https://github.com/Maximiliano0/mPES_2026.git@new_uq


In [11]:
# Resolve pretrained models for ensemble packages — never retrain them here.
# The canonical models ship inside the clone at h1/ml/<pkg>/inputs/. If Drive
# holds newer copies (e.g. from retrain_gpu.ipynb under MyDrive/mPES/<pkg>/),
# they take precedence and are copied over the clone's baseline.
import glob
import os
import shutil

DRIVE_ROOT = '/content/drive/MyDrive/mPES'
ENSEMBLE_MODELS = {'dqn': 'pes_dqn', 'rdqn': 'pes_rdqn', 'trf': 'pes_trf'}

if PKG not in ('ens_sprb', 'ens_accq'):
    print('[INFO] [Celda 3] PKG no es un ensemble; se omite la resolución de modelos.')
else:
    ready = []
    missing = []
    for name, pkg_dir in ENSEMBLE_MODELS.items():
        target = os.path.join(H_DIR, 'ml', pkg_dir, 'inputs', f'{name}_model.keras')
        candidates = []
        for root in (
            os.path.join(DRIVE_ROOT, pkg_dir),
            os.path.join(DRIVE_ROOT, 'h1', 'ml', pkg_dir, 'inputs'),
        ):
            candidates.extend(glob.glob(os.path.join(root, '**', f'{name}*_model.keras'), recursive=True))
            candidates.extend(glob.glob(os.path.join(root, '**', f'{name}_best_*.keras'), recursive=True))
        candidates = [path for path in candidates if os.path.isfile(path)]
        if candidates:
            source = max(candidates, key=os.path.getmtime)
            os.makedirs(os.path.dirname(target), exist_ok=True)
            shutil.copy2(source, target)
            print(f'[INFO] [Celda 3] Modelo desde Drive: {source} → {target}')
        elif os.path.isfile(target):
            print(f'[INFO] [Celda 3] Modelo del clon (baseline): {target}')
        else:
            missing.append(target)
            continue
        ready.append(target)
    if missing:
        raise FileNotFoundError(
            f'Modelos no encontrados ni en Drive ni en el clon: {missing}. '
            'La rama debe incluir los .keras canónicos o Drive debe tener copias.'
        )
    print(f'[INFO] [Celda 3] {len(ready)} modelos listos para {PKG}.')

[INFO] [Celda 3] Modelo del clon (baseline): /content/mPES/h1/ml/pes_dqn/inputs/dqn_model.keras
[INFO] [Celda 3] Modelo del clon (baseline): /content/mPES/h1/ml/pes_rdqn/inputs/rdqn_model.keras
[INFO] [Celda 3] Modelo del clon (baseline): /content/mPES/h1/ml/pes_trf/inputs/trf_model.keras
[INFO] [Celda 3] 3 modelos listos para ens_sprb.


In [ ]:
# Launch the Bayesian optimisation via run_colab.sh (blocks, tails Drive log).
import os
import subprocess
import time

run_environment = os.environ.copy()
run_environment.update({
    'DRIVE_DIR': OUTPUT_ROOT,
    'H_DIR': H_DIR,
    'REPO_DIR': WORKSPACE,
    'WORKSPACE_DIR': WORKSPACE,
    'WORKSPACE': WORKSPACE,
    'PKG': PKG,
    'N_TRIALS': str(N_TRIALS),
    'RESUME_DATE': RESUME_DATE,
    'MPES_USE_GPU': str(USE_GPU),
    'MPES_MODEL_ROOT': '',
})
_script = '''
set -uo pipefail
cd "$WORKSPACE"
source /content/mpes_env.sh
bash "$H_DIR/general/colab/run_colab.sh" "$PKG" "$N_TRIALS" "$RESUME_DATE"
'''
print("[INFO] [Celda 4] Iniciando run_colab.sh... Observa los registros a continuación:")
start = time.time()
run = subprocess.run(
    _script,
    shell=True,
    executable='/bin/bash',
    check=False,
    env=run_environment,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(run.stdout)
elapsed = time.time() - start

# If the run ended suspiciously fast, surface the real error from Drive logs.
if run.returncode != 0 or elapsed < 120:
    run_date = RESUME_DATE or time.strftime('%Y-%m-%d')
    run_dir = os.path.join(OUTPUT_ROOT, f'pes_{PKG}' if not PKG.startswith('pes_') else PKG,
                           f'{run_date}_BAYESIAN_OPT')
    for log_name in ('bayesian_opt_err.log', 'supervisor.log', 'bayesian_opt.log'):
        log_path = os.path.join(run_dir, log_name)
        if os.path.isfile(log_path):
            with open(log_path, 'r', encoding='utf-8', errors='replace') as handle:
                tail = handle.read()[-6000:]
            print(f'\n[WARN] [Celda 4] Últimos registros de {log_name} (terminó en {elapsed:.0f}s):\n{tail}')

if run.returncode != 0:
    print(f"[ERROR] [Celda 4] run_colab.sh falló con código {run.returncode}")
    print(f"[ERROR] Revisa: {OUTPUT_ROOT}/pes_ens_sprb/<FECHA>_BAYESIAN_OPT/")
    raise RuntimeError(f'run_colab.sh exited with code {run.returncode}')
print("[INFO] [Celda 4] Ejecución de run_colab.sh completada con éxito!")

[INFO] [Celda 4] Iniciando run_colab.sh... Observa los registros a continuación:

  Launching Bayesian optimisation on Colab Pro+

  Package         : pes_ens_sprb
  Module          : ens.pes_ens_sprb.ext.optimize_ens
  Trials          : 50
  Run date        : 2026-09-03
  Output dir      : /content/drive/MyDrive/mPES/runs/pes_ens_sprb/2026-09-03_BAYESIAN_OPT
  Storage         : sqlite:////content/drive/MyDrive/mPES/runs/pes_ens_sprb/2026-09-03_BAYESIAN_OPT/optuna_study_2026-09-03.db
  Git             : new_uq@dd00f8a
  Python          : 3.13.15
  GPU mode        : 0

 Optimisation PID    : 17065
 stdout              : /content/drive/MyDrive/mPES/runs/pes_ens_sprb/2026-09-03_BAYESIAN_OPT/bayesian_opt.log
 stderr              : /content/drive/MyDrive/mPES/runs/pes_ens_sprb/2026-09-03_BAYESIAN_OPT/bayesian_opt_err.log
 metadata            : /content/drive/MyDrive/mPES/runs/pes_ens_sprb/2026-09-03_BAYESIAN_OPT/run_meta.json


  Optimisation supervisor launched.
   Cell BLOCKS in foregroun